# Evaluation — final Stirrup work-order trajectory

Evaluate the deterministic one-line answer produced by `07_end_to_end_stirrup_workorder_final.ipynb`. This notebook uses AssetOpsBench's built-in `static_json` scorer rather than an LLM judge. The two headline metrics are pass rate and strict exact match.


In [1]:
from pathlib import Path
import json, os, subprocess, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "evaluation").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
TRACE_DIR = ARTIFACTS / "trajectories"
REPORTS = ARTIFACTS / "reports" / "workorder_static_json"
REPORTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("python:", sys.version.split()[0])


repo: /Users/chathurangishyalika/IBM/AssetOpsBench
python: 3.12.13


In [ ]:
# Load environment variables from .env file in the repository root
from dotenv import load_dotenv

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None
    
repo = find_repo()

if repo:
    load_dotenv(repo / ".env", override=False)

## 1. Materialize the matching static-JSON scenario

The scenario ID must exactly match the ID saved by notebook 07. For work order `NORTH / 1000050`, the accepted one-line reference answer is `Minor in-service problems`. A scalar string is supported by the repository's static-JSON scorer and is normalized case-insensitively before comparison.


In [2]:
SCENARIO_ID = "kdd-workorder-failure-code-001"
EXPECTED_ANSWER = "Minor in-service problems"
SCENARIO_FILE = ARTIFACTS / "kdd_workorder_failure_code_scenario.json"
scenario = {
    "id": SCENARIO_ID,
    "text": (
        "Retrieve work order 1000050 at site NORTH without writing to the database, "
        "then return only its existing or best-fitting failure-code description."
    ),
    "type": "workorder_failure_code",
    "expected_answer": EXPECTED_ANSWER,
    "scoring_method": "static_json",
}
SCENARIO_FILE.write_text(json.dumps(scenario, indent=2) + "\n", encoding="utf-8")
print(SCENARIO_FILE)
print(json.dumps(scenario, indent=2))


/Users/chathurangishyalika/IBM/AssetOpsBench/artifacts/kdd_tutorial/kdd_workorder_failure_code_scenario.json
{
  "id": "kdd-workorder-failure-code-001",
  "text": "Retrieve work order 1000050 at site NORTH without writing to the database, then return only its existing or best-fitting failure-code description.",
  "type": "workorder_failure_code",
  "expected_answer": "Minor in-service problems",
  "scoring_method": "static_json"
}


## 2. Select one valid notebook-07 trajectory

The trajectory directory may contain failed retries with the same scenario ID. Scoring the whole directory would lower pass rate for reasons unrelated to the final run. This cell selects the newest trajectory that contains the exact grounded read, no work-order write calls, and a valid one-line label.


In [3]:
ALLOWED_DESCRIPTIONS = {
    "Breakdown", "Electrical", "Fail to function", "Leaking", "Low output",
    "Minor in-service problems", "Overheating", "Plugged / choked",
    "Structural deficiency", "Vibration",
}
TARGET_INPUT = {"site_id": "NORTH", "wonum": "1000050"}
WRITE_MARKERS = ("generate", "update", "approve", "assign", "close", "cancel", "create", "delete")

def valid_final_trajectory(path):
    try:
        row = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None
    if row.get("scenario_id") != SCENARIO_ID or row.get("runner") != "stirrup-agent":
        return None
    turns = row.get("trajectory", {}).get("turns", [])
    calls = [call for turn in turns for call in turn.get("tool_calls", [])]
    grounded = any(
        call.get("name") == "wo__get_workorder" and call.get("input") == TARGET_INPUT
        for call in calls
    )
    write_calls = [
        call.get("name") for call in calls
        if str(call.get("name", "")).startswith("wo__")
        and any(marker in str(call.get("name", "")) for marker in WRITE_MARKERS)
    ]
    answer = str(row.get("answer", "")).strip()
    if not grounded or write_calls or "\n" in answer or answer not in ALLOWED_DESCRIPTIONS:
        return None
    return row

candidates = []
for path in TRACE_DIR.glob("*.json"):
    row = valid_final_trajectory(path)
    if row is not None:
        candidates.append((path.stat().st_mtime, path, row))
assert candidates, "No valid final work-order trajectory found. Run notebook 07 successfully first."
_, SELECTED_TRAJECTORY, selected = max(candidates, key=lambda item: item[0])
print("selected trajectory:", SELECTED_TRAJECTORY)
print("run_id:", selected["run_id"])
print("answer:", selected["answer"])


selected trajectory: /Users/chathurangishyalika/IBM/AssetOpsBench/artifacts/kdd_tutorial/trajectories/kdd-stirrup-wo-20260805T131613Z.json
run_id: kdd-stirrup-wo-20260805T131613Z
answer: Minor in-service problems


## 3. Run the repository's static-JSON evaluator

No judge model or model credentials are required. `StaticJsonScorer` marks a scenario as passed only when `strict_exact_match_accuracy == 1.0`.


In [4]:
cmd = [
    "uv", "run", "--directory", str(REPO), "evaluate",
    "--trajectories", str(SELECTED_TRAJECTORY),
    "--scenarios", str(SCENARIO_FILE),
    "--reports-dir", str(REPORTS),
    "--scorer-default", "static_json",
]
completed = subprocess.run(cmd, text=True, capture_output=True, env=os.environ.copy())
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
completed.check_returncode()


Scenarios: 1  Passed: 1  Pass rate: 100.0%

Score summary by runner/model:
  stirrup-agent_litellm_proxy/aws/claude-opus-4-8:
    score_avg: 1.0000
    score_min: 1.0000
    score_max: 1.0000
    partial_match_avg: 1.0000
    partial_exact_match_avg: 1.0000
    strict_exact_match_avg: 1.0000
    partial_similarity_avg: 1.0000
    partial_numeric_match_avg: 0.0000
    range_match_avg: 0.0000
    delta_1_match_avg: 0.0000
    precision_avg: 1.0000
    recall_avg: 1.0000
    f1_avg: 1.0000
    total_gold_keys_avg: 1.0000
    total_model_keys_avg: 1.0000
    matched_keys_avg: 1.0000
    exact_value_matches_avg: 1.0000
    missing_keys_total: 0
    extra_keys_total: 0
    detail_entries_total: 1

By scenario type:
  workorder_failure_code    1/1     (100.0%)

Operational metrics:
  tokens_in_total:   61332
  tokens_out_total:  181
  tool_calls_total:  2
  duration_ms_p50:   30181.8
  duration_ms_p95:   30181.8

Aggregate report written: /Users/chathurangishyalika/IBM/AssetOpsBench/artifacts

## 4. Report pass rate and exact match

`pass_rate` is `passed / scored`. `strict_exact_match_accuracy` is the deterministic exact-match metric emitted by `src/evaluation/scorers/static_json.py`. For one correct trajectory, both values are `1.0` (100%).


In [5]:
aggregate_path = REPORTS / "_aggregate.json"
assert aggregate_path.exists(), f"Missing evaluator report: {aggregate_path}"
aggregate = json.loads(aggregate_path.read_text(encoding="utf-8"))
assert aggregate.get("results"), "The scenario and trajectory did not join; check scenario_id."
result = aggregate["results"][0]
details = result.get("score", {}).get("details", {})
metrics = {
    "scored": aggregate.get("totals", {}).get("scored", 0),
    "passed": aggregate.get("totals", {}).get("passed", 0),
    "pass_rate": aggregate.get("totals", {}).get("pass_rate", 0.0),
    "exact_match": details.get("strict_exact_match_accuracy", 0.0),
    "model_answer": result.get("answer"),
    "expected_answer": EXPECTED_ANSWER,
}
print(json.dumps(metrics, indent=2))


{
  "scored": 1,
  "passed": 1,
  "pass_rate": 1.0,
  "exact_match": 1.0,
  "model_answer": "Minor in-service problems",
  "expected_answer": "Minor in-service problems"
}


In [6]:
assert metrics["scored"] == 1
assert metrics["pass_rate"] == 1.0
assert metrics["exact_match"] == 1.0
assert metrics["model_answer"] == EXPECTED_ANSWER
print("EVALUATION PASSED: pass rate = 100%, exact match = 100%")


EVALUATION PASSED: pass rate = 100%, exact match = 100%
